# Step 1 — Exploratory Data Analysis (EDA)

| | |
|---|---|
| **Goal** | Understand the data *before* changing it, and collect the **evidence** that will justify every preprocessing decision. |
| **Inputs** | `data/interim/<day>.parquet` (step 0) |
| **Outputs** | Tables in `results/tables/01_*.csv`, figures in `results/figures/01_*.png`, and a list of preprocessing decisions |
| **Tools** | pandas, PyArrow (streamed reading), NumPy, matplotlib / seaborn |

### Why EDA matters so much here
The course rubric says: *"a model with 99% accuracy and no critical analysis will score lower than a modest model
supported by a rigorous methodology"*. CIC-IDS datasets are famous for giving 99%+ accuracy **for the wrong
reasons**: duplicates shared by train and test, artefact features, mislabelled flows. EDA is where we find these
problems, so that our results mean something.

### What step 0 already told us
| Observation | Why it matters |
|---|---|
| 7 of 10 files have **exactly 1,048,575 flows** (+1 header line = 1,048,576) | That is **Excel's maximum row count**. These CSVs were very likely saved through Excel and **truncated**: part of those days is missing. This is a dataset limitation to report. |
| Timestamps from **1970** in the 14/02 and 22/02 files | Invalid timestamps (Unix epoch). Must be handled before a time-based split. |
| Every day runs from 01:00 to 12:59 | Suggests a **12-hour clock without AM/PM**. We test this hypothesis below against the official attack schedule. |
| No `DDOS attack-LOIC-UDP` flow on 20/02, although it is documented (13:13–13:32) | Either missing from the CSV or mislabelled as benign. Another limitation. |
| SQL Injection: **87 flows** vs 13.5 M benign flows (ratio ≈ 155,000 : 1) | This is the extreme imbalance that motivates the GAN. But it also limits how reliably we can *evaluate* these classes. |

### Method: streamed audit + stratified sample
16 M flows × 78 features in float64 ≈ **10 GB**, more than Colab's RAM. So:
1. **Exact statistics on 100% of the data** with a single streamed pass (500k rows at a time): missing/infinite
   values, min/max, timestamps, and a **64-bit hash of each flow** to find duplicates without keeping the rows in memory.
2. **A stratified sample** (up to 30,000 flows per class) for everything that needs the actual values (distributions,
   correlations). It is *stratified* because a plain 1% random sample would contain about one SQL Injection flow.

## 1. Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/dija040/ids-project.git"
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/ids-project")

    def git(*args):
        # Run a git command; if it fails, show git's own error message.
        r = subprocess.run(["git", *args], capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError("git " + " ".join(args) + " failed: " + r.stderr)
        return r.stdout.strip()

    if not (PROJECT_DIR / ".git").exists():
        git("clone", REPO_URL, str(PROJECT_DIR))
    else:
        # The Drive copy is only used to RUN the code (we edit it on the PC and push to
        # GitHub), so we force it to match GitHub exactly instead of merging with `pull`.
        # Untracked files (data/, generated results) are not touched.
        git("-C", str(PROJECT_DIR), "fetch", "origin")
        git("-C", str(PROJECT_DIR), "reset", "--hard", "origin/main")
    print("Code version:", git("-C", str(PROJECT_DIR), "log", "-1", "--format=%h %s"))
else:
    PROJECT_DIR = Path.cwd().resolve().parent
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import json, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from src import config, eda
config.set_seed(); config.ensure_dirs()
pd.set_option("display.max_rows", 100, "display.width", 200)
FIG, TAB = config.FIGURES_DIR, config.TABLES_DIR

## 2. Inventory check: is the Excel truncation real?

If a file was cut at Excel's limit, then `rows + header lines removed + 1 (header)` is exactly **1,048,576**.

In [ ]:
summaries = {d: json.loads((config.INTERIM_DIR / f"{d}.summary.json").read_text())
             for d in config.DAY_FILES}
inv = pd.DataFrame(summaries).T[["rows", "header_rows_removed"]].astype(int)
inv["csv_lines"] = inv["rows"] + inv["header_rows_removed"] + 1
inv["excel_truncated"] = inv["csv_lines"] == config.EXCEL_MAX_ROWS
print(f"Truncated files: {inv['excel_truncated'].sum()} / {len(inv)}")
inv

**How to read this.** A truncated file does not tell us *what* was cut. CSV rows are not in chronological order
(we saw that in step 0), so the missing part is not simply "the end of the day". The consequences are:
- the per-day class counts are **not the true attack volumes**;
- we **cannot recover** the missing flows from this version of the dataset (only by re-extracting the PCAPs);
- in the report, this goes in *Dataset → known limitations*, ideally with a reference to the literature.

## 3. Streamed audit of the full dataset

One pass over the 10 days. ⏱️ About 10–20 minutes on Colab (the 20/02 file is ~half of the data).

In [ ]:
label_totals = {}
for s in summaries.values():
    for lab, n in s["label_counts"].items():
        label_totals[lab] = label_totals.get(lab, 0) + n
day_paths = {d: config.INTERIM_DIR / f"{d}.parquet" for d in config.DAY_FILES}

t0 = time.time()
A = eda.audit_dataset(day_paths, label_totals, sample_per_class=30_000)
print(f"Audit done in {time.time()-t0:.0f}s | hashed flows: {len(A['hashes']):,} | sample: {len(A['sample']):,} flows")

# Keep the sample: it is reused at the end of this notebook and is cheap to store (~200 MB).
A["sample"].to_parquet(config.INTERIM_DIR / "eda_sample.parquet", index=False)

## 4. Data quality per feature: missing, infinite, negative, constant

What we are looking for, and why:
- **NaN / ±inf:** most ML models crash on them, so we must decide how to handle them. *Where do they come from?*
  CICFlowMeter computes rates such as `Flow Byts/s = bytes / duration`. When the duration is 0, the result is a
  division by zero, giving `inf` (or `NaN` for 0/0). This tells us the values are **not random noise**: they flag
  very short flows. That will influence the decision (drop the rows? replace the values? add an indicator?).
- **Negative values** in quantities that cannot be negative (durations, counts, times between packets) reveal
  **extraction bugs**. One exception: `Init Fwd/Bwd Win Byts = -1` is a **sentinel** meaning "not observed".
- **Constant features** (min == max over 16 M flows) carry no information. We can drop them safely.

In [ ]:
q = A["quality"]
q.to_csv(TAB / "01_feature_quality.csv")
print("Features with NaN or inf:")
display(q[(q["nan"] + q["+inf"] + q["-inf"]) > 0][["nan", "+inf", "-inf"]])
print("Features with negative values:")
display(q[q["negative"] > 0][["negative", "min", "max"]])
const = q.index[q["constant"]].tolist()
print(f"{len(const)} constant features (min == max on all 16 M flows):", const)

In [ ]:
# Who are the flows with NaN/inf? If they are concentrated in some classes, dropping them is NOT neutral.
s = A["sample"]
bad = ~np.isfinite(s[config.FEATURE_COLUMNS].to_numpy()).all(axis=1)
nonfinite_by_class = (pd.DataFrame({"sample_flows": s.groupby("Label").size(),
                                    "with_nan_or_inf": s[bad].groupby("Label").size()})
                        .fillna(0).astype(int))
nonfinite_by_class["percent"] = (100 * nonfinite_by_class["with_nan_or_inf"] / nonfinite_by_class["sample_flows"]).round(2)
nonfinite_by_class.sort_values("percent", ascending=False)

## 5. Duplicates and conflicting labels (on all 16 M flows)

Two different problems:
- **Exact duplicates** (same 78 features, same label). With a random split, copies of the same flow end up in
  **both train and test**, so the model is evaluated on flows it has memorised. This is the classic cause of the
  inflated scores on CIC-IDS. Our time-aware split reduces this risk but does not remove it, because identical flows
  can occur at different times.
- **Conflicting duplicates** (same features, *different* labels). No model can classify both correctly. This is
  **direct evidence of label noise**. The literature reports, for example, that many flows labelled as attacks are in
  fact benign-looking background traffic (e.g. TCP connection attempts that captured nothing malicious).

In [ ]:
t0 = time.time()
dups, pairs = eda.duplicate_report(A["hashes"])
print(f"computed in {time.time()-t0:.0f}s")
total_dup = dups["exact_duplicates"].sum()
print(f"Exact duplicates: {total_dup:,} flows ({100*total_dup/dups['flows'].sum():.1f}% of the dataset)")
dups.to_csv(TAB / "01_duplicates_per_class.csv"); pairs.to_csv(TAB / "01_conflicting_label_sets.csv")
display(dups)
print("Label sets that share identical feature vectors (top 15):")
display(pairs.head(15))

**Questions to think about:**
- Which classes are the **most duplicated**? DoS/DDoS tools send the same request again and again, so duplicates
  are expected there. Is that a "bug", or the real nature of the attack? Should we keep one copy, or all of them?
- What is the size of the **conflicts**? If an attack class shares many feature vectors with `Benign`, its recall has
  a hard ceiling, whatever the model (and whatever the GAN).

## 6. Time: testing the 12-hour-clock hypothesis

**Hypothesis H:** timestamps were written on a 12-hour clock without AM/PM, so 13:00–17:00 appears as 01:00–05:00.

**Test:** if H is true, each attack documented in the afternoon should appear in the data **12 hours earlier**
(e.g. SSH-Bruteforce, documented 14:01–15:31, should appear at 02:01–03:31), and morning attacks should match.

In [ ]:
cmp = eda.documented_vs_observed(A["windows"])
def h(t):  # "HH:MM" -> hour as int
    return int(t[:2]) if isinstance(t, str) else np.nan
cmp["doc_start_on_12h_clock"] = cmp["doc_start"].map(lambda t: f"{h(t) % 12 or 12:02d}{t[2:]}" if isinstance(t, str) else t)
cmp.to_csv(TAB / "01_attack_windows_documented_vs_observed.csv", index=False)
cmp

In [ ]:
# Flows per hour of the (raw) clock, for each day: holes and peaks are visible at a glance.
hours = A["hours"]
pivot_all = hours.groupby(["day", "hour"])["flows"].sum().unstack(fill_value=0)
pivot_att = hours[hours["Label"] != "Benign"].groupby(["day", "hour"])["flows"].sum().unstack(fill_value=0)
pivot_att = pivot_att.reindex(index=pivot_all.index, columns=pivot_all.columns, fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, pv, title in [(axes[0], pivot_all, "All flows"), (axes[1], pivot_att, "Attack flows only")]:
    sns.heatmap(np.log10(1 + pv), ax=ax, cmap="viridis", cbar_kws={"label": "log10(1 + flows)"})
    ax.set_title(f"{title} per hour of the recorded timestamp"); ax.set_xlabel("hour (as written in the CSV)")
plt.tight_layout(); plt.savefig(FIG / "01_flows_per_hour.png", dpi=150); plt.show()

In [ ]:
bt = A["bad_timestamps"]
print(f"{len(bt):,} flows with a timestamp outside 2018")
if len(bt):
    display(bt.groupby(["day", "Label"]).size().rename("flows").to_frame())
    display(bt.head())

**Interpretation (fill in after running):** does the table support H? If it does, then *sorting by timestamp*
puts the afternoon traffic **before** the morning traffic, and a naive time-aware split would be wrong.
Possible fix in preprocessing: rebuild the real time using the attack schedule (hour + 12 for the afternoon block).
We need to find a rule that does not use the labels, otherwise we leak label information into the time column.
We will design that rule together in step 2.

## 7. Imbalance: what it means for training *and* for evaluation

We already know the ratios. Here is the consequence that is easy to miss: with a 70 / 15 / 15 split,
how many flows of each class will be left in **validation** and **test**?

In [ ]:
tot = pd.Series(label_totals).sort_values()
split = pd.DataFrame({"total": tot, "train_70%": (tot * .70).astype(int),
                      "val_15%": (tot * .15).astype(int), "test_15%": (tot * .15).astype(int)})
split.head(6)

With only ~13 SQL Injection flows in the test set, **one** extra detected flow changes recall by ~8 points.
Scores on these classes will have very wide confidence intervals. Two options to discuss in step 2:
1. **Keep the classes and report uncertainty** (e.g. bootstrap confidence intervals).
2. **Group them into attack families** (e.g. `Web attack` = Brute Force-Web + XSS + SQL Injection ≈ 930 flows).
   This is more stable, still meaningful for a SOC, and more realistic for the GAN (which needs enough examples to learn from).

Remember also: the GAN learns **only from the training split**. For SQL Injection, that means about **60 real flows**.
This is a real limit on what any generative model can learn.

## 8. Feature distributions per class (sample)

Network features are **heavy-tailed**: most flows are tiny, a few are huge. On a linear scale everything collapses
near zero, so we use `signed_log(x) = sign(x)·log10(1+|x|)`. The question to ask: **which features visibly separate
classes**, and which classes look alike (they will be confused)?

In [ ]:
s = A["sample"].replace([np.inf, -np.inf], np.nan)
order = s["Label"].value_counts().index
show = ["Flow Duration", "Tot Fwd Pkts", "TotLen Fwd Pkts", "Flow Byts/s",
        "Flow IAT Mean", "Pkt Len Mean", "Init Fwd Win Byts", "Fwd Seg Size Min"]
fig, axes = plt.subplots(4, 2, figsize=(16, 22))
for ax, col in zip(axes.ravel(), show):
    sns.boxplot(y=s["Label"], x=eda.signed_log(s[col]), order=order, ax=ax,
                showfliers=False, color="#8DA0CB")
    ax.set_title(col); ax.set_xlabel("signed log10"); ax.set_ylabel("")
plt.tight_layout(); plt.savefig(FIG / "01_feature_distributions.png", dpi=120); plt.show()

## 9. Redundancy: correlated features

Many CICFlowMeter features are **arithmetic variants of one another** (e.g. `Subflow Fwd Pkts` vs `Tot Fwd Pkts`,
`Pkt Len Var` = `Pkt Len Std`²). Redundant features slow training, blur feature importance (it gets split between
twins), and add nothing. We use the **Spearman** correlation (based on ranks): it is robust to heavy tails and
catches any monotonic relation, not only linear ones.

In [ ]:
num = s[[c for c in config.FEATURE_COLUMNS if c not in const]]
corr = num.corr(method="spearman")
upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
high = (upper.stack().rename("rho").reset_index()
             .rename(columns={"level_0": "feature_a", "level_1": "feature_b"}))
high = high[high["rho"].abs() >= 0.95].sort_values("rho", key=abs, ascending=False)
high.to_csv(TAB / "01_highly_correlated_pairs.csv", index=False)
print(f"{len(high)} pairs with |rho| >= 0.95 among {num.shape[1]} non-constant features")
display(high.head(25))

plt.figure(figsize=(16, 14))
sns.heatmap(corr, cmap="coolwarm", center=0, xticklabels=True, yticklabels=True)
plt.xticks(fontsize=6); plt.yticks(fontsize=6); plt.title("Spearman correlation (stratified sample)")
plt.tight_layout(); plt.savefig(FIG / "01_correlation_heatmap.png", dpi=150); plt.show()

## 10. Destination port: a feature, or a shortcut?

`Dst Port` is very predictive: FTP brute force targets port 21, SSH brute force port 22, web attacks port 80.
But that is because **this testbed** ran each service on its standard port. Two views to debate:
- **Keep it:** ports carry real meaning, and real SOCs use them too.
- **Drop it, or test without it:** a model that has learned "port 22 ⇒ SSH attack" will flag every SSH connection
  and will miss an attack on a non-standard port (an easy evasion).

A strong report can measure both. Look at how concentrated each class is:

In [ ]:
top = (s.groupby("Label")["Dst Port"].value_counts(normalize=True)
         .rename("share").reset_index()
         .groupby("Label").head(3))
top["share"] = (100 * top["share"]).round(1)
top.pivot_table(index="Label", columns=top.groupby("Label").cumcount() + 1,
                values=["Dst Port", "share"], aggfunc="first")

## 11. Summary: from findings to decisions

Fill in the **evidence** column with your numbers. We will discuss the **decision** column together, and it will
then be implemented in step 2 (preprocessing).

| # | Finding | Evidence (your numbers) | Decision options (to decide in step 2) |
|---|---|---|---|
| 1 | Excel truncation | __ files at 1,048,575 rows | Report as a limitation; per-day counts are not true volumes |
| 2 | Invalid timestamps (1970) | __ flows | Drop them / infer the time from their neighbours in the file |
| 3 | 12-hour clock | table §6 | Rebuild the time of day / split on `row_in_file` order / split by attack window |
| 4 | NaN / inf | __ flows, in classes __ | Drop rows / impute with a cap / add an indicator feature |
| 5 | Negative values | features __ | Clip to 0 / treat as missing / keep the -1 sentinels |
| 6 | Constant features | __ features | Drop |
| 7 | Exact duplicates | __ % | Drop before the split / keep (attack nature) / drop within each split |
| 8 | Conflicting labels | __ vectors | Remove / relabel / keep and report as noise |
| 9 | Very rare classes | 87 / 230 / 611 flows | Keep with confidence intervals / group into families |
| 10 | Redundant features | __ pairs ≥ 0.95 | Handled in step 3 (feature selection) |
| 11 | `Dst Port` shortcut | §10 | Experiment with and without |
| 12 | LOIC-UDP missing on 20/02 | 0 flows | Report as a limitation |

> 📝 Copy your numbers into `reports/step_reports/01_eda.md`, and send me the outputs of sections 2, 4, 5, 6 and 10.

## Publish the results to GitHub

Commits the tables and figures of this notebook (`results/`) and pushes them to GitHub, so they are
available on the PC and in the repository for the report. Needs the one-time `GITHUB_TOKEN` secret
(see the README, or the instructions printed below if it is missing). Data files are never published.

In [ ]:
from src.publish import publish_results
publish_results("Results of notebook 01 (Colab run)")